# Solar Filament Segmentation 2026: end-to-end pipeline

This notebook reproduces the submitted predictions from the files in `weights/`: ten U-Nets (EfficientNetV2-S and EfficientNet-B3 encoders, one per cross-validation fold each), the decision-layer scorer and five instance refiners. The method is described in `README.md` and `report/main.pdf`. Re-training is optional and listed at the end.

Requirements: `pip install -r requirements.txt`, and the Kaggle data unzipped into `data/raw/MAGFiLO_1.0_Kaggle_2026/`. A CUDA GPU with 8 GB is enough.

In [ ]:
import sys, os, glob, json, pickle
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, 'src')
import numpy as np, pandas as pd, cv2, torch, matplotlib.pyplot as plt
from common import RAW, load_image, normalize, norm_stats
from predict import load_models, predict_prob
from decision import process_image, select, _score
from refine import load_refiners, refine_probs, assemble
PARAMS = json.load(open('weights/params.json'))
FAMILIES = {f: sorted(glob.glob(f'weights/{f}_f*.pt')) for f in PARAMS['families']}   # order matters: as in the scorer's bank
with open(PARAMS['scorer'], 'rb') as fh:
    SCORER = pickle.load(fh)
REFINERS = sorted(glob.glob('weights/refiner_f*.pt'))
print(PARAMS, {k: len(v) for k, v in FAMILIES.items()}, len(REFINERS), 'refiners, cuda:', torch.cuda.is_available())

## 1. Data and targets
Training annotations are COCO polygons from 1 to 3 annotators per image. `src/prepare.py` rasterises them into per-annotator instance maps and three consensus targets: the soft filament mask (fraction of annotators), a normalised distance-to-boundary map and the contact line between touching filaments. Folds are grouped by observation date.

In [ ]:
if not os.path.exists('data/masks/touch'):
    !python src/prepare.py
splits = pd.read_csv('data/splits.csv'); print(splits.fold.value_counts().sort_index().to_dict(), 'files per fold')
stem = splits[splits.fold == 0].stem.iloc[3]
img = load_image(f'{RAW}/train/train_images/{stem}.jpeg')
soft = cv2.imread(f'data/masks/soft/{stem}.png', 0)
fig, ax = plt.subplots(1, 2, figsize=(14, 7)); ax[0].imshow(img, cmap='gray'); ax[0].set_title(stem); ax[1].imshow(soft, cmap='magma'); ax[1].set_title('consensus mask')
for a in ax: a.axis('off')
plt.show()

## 2. Dense prediction
Each model predicts the whole 2048 x 2048 frame, averaged over the eight flips/rotations; the five fold models of each encoder family are averaged, then the two families. Output channels: filament probability, distance map, contact line.

In [ ]:
test_paths = sorted(glob.glob(f'{RAW}/test/test_images/*.jpeg'))
example = os.path.basename(test_paths[0])[:-5]; img = load_image(test_paths[0]); demo = []
for fam, ws in FAMILIES.items():
    models = load_models(ws, 'cuda')
    out = predict_prob(models, img, 'cuda', tta=8)
    os.makedirs(f'runs/maps/demo_{fam}', exist_ok=True); demo.append(f'runs/maps/demo_{fam}/{example}.npy')
    np.save(demo[-1], (out * 255).round().astype(np.uint8))
    del models; torch.cuda.empty_cache()
p = np.round(np.mean([np.load(f)[0] for f in demo], 0)).astype(np.float32) / 255
fig, ax = plt.subplots(1, 2, figsize=(14, 7)); ax[0].imshow(img, cmap='gray'); ax[0].set_title(example); ax[1].imshow(p, cmap='magma'); ax[1].set_title('mean filament probability of 10 models')
for a in ax: a.axis('off')
plt.show()

## 3. Decision layer: component tree, scorer, exact selection
The probability map is thresholded at 16 levels; the nested connected components form a forest of candidates. A gradient-boosted hurdle model predicts each candidate's expected matched IoU against a random annotator, and a dynamic programme selects the disjoint set that maximises the sum of (score - tau). Idea credited to the public notebook by _.tushhaaarrrr (see README). The full run in section 6 picks tau so that the instances per image match the cross-validated rate; here the cross-validated tau is used.

In [ ]:
r = process_image(example, demo, test_paths[0], keep_labels=True)
nd = r['nodes']; u = _score(SCORER['model'], nd[SCORER['features']].to_numpy(np.float32))
chosen = sorted(select(nd.parent.to_numpy(), u - SCORER['tau']), key=lambda v: -u[v])     # descending score
seeds = [r['labels'][int(nd.level_idx.iloc[v])] == int(nd.lab_id.iloc[v]) for v in chosen]
print(f"{len(nd)} candidates, {len(chosen)} selected at tau = {SCORER['tau']}")

## 4. Stage 2: instance refinement
Every selected instance is re-segmented by a UNet++ that sees a native-resolution window around it (image, probability, the instance, the other selected instances) and predicts the outline an annotator would draw given that they drew this filament. Refined masks stay pixel-disjoint.

In [ ]:
refiners = load_refiners(REFINERS)
refined = assemble(seeds, refine_probs(refiners, normalize(img, norm_stats(img, 'disk')), p, seeds), PARAMS['refiner_threshold'])
ys, xs = np.nonzero(np.any(seeds, 0)); cy, cx = int(np.median(ys)), int(np.median(xs))
sl = (slice(max(cy - 400, 0), cy + 400), slice(max(cx - 400, 0), cx + 400))
view = cv2.cvtColor((img[sl] * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB); a, b = view.copy(), view.copy()
for masks, canvas, colour in ((seeds, a, (255, 170, 0)), (refined, b, (0, 220, 255))):
    for m in masks:
        cs, _ = cv2.findContours(np.ascontiguousarray(m[sl]).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(canvas, cs, -1, colour, 1)
fig, ax = plt.subplots(1, 2, figsize=(14, 7)); ax[0].imshow(a); ax[0].set_title('decision layer (stage 1)'); ax[1].imshow(b); ax[1].set_title('after refinement (stage 2)')
for x in ax: x.axis('off')
plt.show()
del refiners; torch.cuda.empty_cache()

## 5. Validation summary (out-of-fold)
Pooled PQ as the host computes it, per fold: plain rule, decision layer, decision layer + refiner, and the candidate oracle. These numbers come from `src/decision.py fit`, `src/refine.py eval` and `scripts/make_decision_table.py`; recomputing them needs the out-of-fold maps (see section 7).

In [ ]:
summary = json.load(open('report/decision_summary.json'))
pd.DataFrame({k: summary[k] for k in ['0', '1', '2', '3', '4', 'all']}).T[['baseline', 'decision', 'refined', 'oracle']].round(4)

## 6. Test predictions and submission file
All 180 test frames: cache the dense maps of both families, run the decision layer and the refiner, and verify the CSV (RLE decodability, unique ids, no two masks of an image overlap). About 50 minutes on a laptop GPU; the submitted file is `report/submission_final.csv`.

In [ ]:
!bash scripts/predict_test.sh submission.csv
pd.read_csv('submission.csv').head()

## 7. (Optional) Re-train and re-fit
```
python src/prepare.py
scripts/train_folds.sh efficientnet-b3 effb3d "0 1 2 3 4" --epochs 50 --crop 640 --bs 6 --eval_every 10 --norm fixed
scripts/train_folds.sh tu-tf_efficientnetv2_s effv2s "0 1 2 3 4" --epochs 75 --crop 512 --bs 16 --eval_every 25 --norm disk
for k in 0 1 2 3 4; do python src/export_weights.py runs/effb3d_f$k weights/effb3d_f$k.pt; python src/export_weights.py runs/effv2s_f$k weights/effv2s_f$k.pt --ckpt last.pt; done
for f in effv2s effb3d; do python src/predict_maps.py oof --weights weights/${f}_f*.pt --out runs/maps/oof_$f --tta 8; done
python src/decision.py bank --maps runs/maps/oof_effv2s runs/maps/oof_effb3d --out runs/decision/bank_v2s_b3.pkl
python src/decision.py fit --bank runs/decision/bank_v2s_b3.pkl --out runs/decision/scorer_v2s_b3.pkl
python src/refine.py seeds --maps runs/maps/oof_effv2s runs/maps/oof_effb3d --scores runs/decision/scorer_v2s_b3_oof_scores.pkl --scorer runs/decision/scorer_v2s_b3.pkl --out runs/refine/v2s_b3
for k in 0 1 2 3 4; do python src/refine.py train --seeds runs/refine/v2s_b3 --fold $k --arch UnetPlusPlus --out weights/refiner_f$k.pt; done
python src/refine.py eval --seeds runs/refine/v2s_b3 --weights weights/refiner_f*.pt --json runs/refine/v2s_b3/eval.json
cp runs/decision/scorer_v2s_b3.pkl weights/decision_scorer.pkl
python scripts/make_decision_table.py && python scripts/make_report_figures.py
```